# 08 — Macro Model Development

This notebook develops and compares forecasting models that extend the baseline model with macroeconomic and industry-level indicators.

The objective is to evaluate whether external drivers improve monthly car registration forecasts beyond historical registration patterns alone.

The notebook explores several model variants, including:

- Linear regression with macroeconomic indicators
- Standardized macro models for coefficient interpretation
- Feature-selected macro models
- Ridge regression with regularization
- Random Forest and XGBoost models for non-linear relationships

The final output is a comparison of candidate models and the selection of a preferred model for the 2026 forecasting workflow.

## Development Note

Several candidate models are tested to compare interpretability, forecast accuracy, and suitability for scenario-based forecasting.

The final production-ready model is cleaned and saved separately for use in the forecasting notebook.

In [ ]:
%pip install xgboost

In [ ]:
# =========================================================
#  1. Import Required Libraries
# =========================================================

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_absolute_error, mean_squared_error

In [ ]:
# =========================================================
# Load datasets
# =========================================================
macro_df = pd.read_parquet("data/macro_df.parquet")

In [ ]:
# =========================================================
#  2. Prepare the Dataset
# =========================================================

macro_df = macro_df.copy()
macro_df["date"] = pd.to_datetime(macro_df["date"])
macro_df = macro_df.sort_values("date").reset_index(drop=True)

print(macro_df.head())
print(macro_df.dtypes)

## Model 1 — Linear Regression with Macro Indicators

This model extends the baseline approach by incorporating macroeconomic indicators alongside historical registration data.

The objective is to evaluate whether external economic factors improve predictive performance compared to using registration history alone.

The model includes:

- Lagged registration features
- Rolling averages
- Seasonal features (month-based encoding)
- Selected macroeconomic indicators (e.g. CPI, interest rates, fuel prices)

This serves as the first step in assessing the contribution of macro variables to forecasting accuracy.

In [ ]:
# =========================================================
#  3. Define Target and Features
# =========================================================

TARGET = "acea_total_registrations"

FEATURES = [col for col in macro_df.columns if col not in ["date", TARGET]]

print("Target:", TARGET)
print("Number of features:", len(FEATURES))
print("Features:")
print(FEATURES)

In [ ]:
# =========================================================
#  4. Check for Missing Values
# =========================================================

missing_summary = macro_df[FEATURES + [TARGET]].isna().sum()
print(missing_summary[missing_summary > 0])

In [ ]:
# =========================================================
#  5. Build the Macro Model
# =========================================================

model = LinearRegression()

In [ ]:
# =========================================================
#  6. Define the Rolling Backtesting Function
# =========================================================

def rolling_backtest(
    df,
    start_date,
    features,
    target,
    model
):
    df = df.copy()
    df["date"] = pd.to_datetime(df["date"])
    df = df.sort_values("date").reset_index(drop=True)

    start_date = pd.to_datetime(start_date)

    predictions = []
    actuals = []
    dates = []

    for i in range(len(df)):
        current_date = df.loc[i, "date"]

        if current_date < start_date:
            continue

        train = df[df["date"] < current_date]
        test = df[df["date"] == current_date]

        if len(train) == 0 or len(test) == 0:
            continue

        X_train = train[features]
        y_train = train[target]

        X_test = test[features]
        y_test = test[target]

        model.fit(X_train, y_train)
        y_pred = model.predict(X_test)

        predictions.append(y_pred[0])
        actuals.append(y_test.values[0])
        dates.append(current_date)

    results = pd.DataFrame({
        "date": dates,
        "actual": actuals,
        "prediction": predictions
    })

    return results

In [ ]:
# =========================================================
#  7. Run Rolling Backtesting
# =========================================================

macro_results = rolling_backtest(
    df=macro_df,
    start_date="2024-01-01",
    features=FEATURES,
    target=TARGET,
    model=model
)

print(macro_results.head())
print(macro_results.tail())

In [ ]:
# =========================================================
#  8. Evaluate Macro Model Performance
# =========================================================

macro_mae = mean_absolute_error(macro_results["actual"], macro_results["prediction"])
macro_rmse = np.sqrt(mean_squared_error(macro_results["actual"], macro_results["prediction"]))
macro_mape = np.mean(
    np.abs((macro_results["actual"] - macro_results["prediction"]) / macro_results["actual"])
) * 100

print(f"Macro Model MAE: {macro_mae:,.2f}")
print(f"Macro Model RMSE: {macro_rmse:,.2f}")
print(f"Macro Model MAPE: {macro_mape:.2f}%")

In [ ]:
# =========================================================
#  Visualize Results
# =========================================================

import matplotlib.pyplot as plt

plt.figure(figsize=(12,6))
plt.plot(macro_results["date"], macro_results["actual"], label="Actual")
plt.plot(macro_results["date"], macro_results["prediction"], label="Prediction")
plt.legend()
plt.title("Macro Model - Rolling Forecast")
plt.show()

In [ ]:
# =========================================================
#  9. Create a Detailed Results Table
# =========================================================

macro_results["error"] = macro_results["prediction"] - macro_results["actual"]
macro_results["abs_error"] = np.abs(macro_results["error"])
macro_results["error_pct"] = (macro_results["abs_error"] / macro_results["actual"]) * 100

macro_results_table = macro_results[[
    "date",
    "actual",
    "prediction",
    "error",
    "abs_error",
    "error_pct"
]].copy()

macro_results_table["actual"] = macro_results_table["actual"].round(0)
macro_results_table["prediction"] = macro_results_table["prediction"].round(0)
macro_results_table["error"] = macro_results_table["error"].round(0)
macro_results_table["abs_error"] = macro_results_table["abs_error"].round(0)
macro_results_table["error_pct"] = macro_results_table["error_pct"].round(2)

print(macro_results_table)

In [ ]:
# =========================================================
#  11. Compare Macro Model vs Baseline Model
# =========================================================

comparison_table = pd.DataFrame({
    "model": ["Baseline Model", "Macro Model"],
    "MAE": [64813.86, macro_mae],
    "RMSE": [83416.23, macro_rmse],
    "MAPE": [7.59, macro_mape]
})

comparison_table["MAE"] = comparison_table["MAE"].round(2)
comparison_table["RMSE"] = comparison_table["RMSE"].round(2)
comparison_table["MAPE"] = comparison_table["MAPE"].round(2)

print(comparison_table)

In [ ]:
# =========================================================
#  Extract Model Coefficients
# =========================================================

coef_df = pd.DataFrame({
    "feature": FEATURES,
    "coefficient": model.coef_
})

coef_df["abs_coef"] = coef_df["coefficient"].abs()

# Sort by importance
coef_df = coef_df.sort_values("abs_coef", ascending=False)

display(coef_df)

In [ ]:
print("Intercept:", model.intercept_)

In [ ]:
from sklearn.metrics import r2_score

# Fit model on full dataset (for interpretation only)
model.fit(macro_df[FEATURES], macro_df[TARGET])

y_pred_full = model.predict(macro_df[FEATURES])

r2 = r2_score(macro_df[TARGET], y_pred_full)

print(f"R² (in-sample): {r2:.4f}")

## Model 2 — Standardized Linear Regression

This model builds on the previous macro model by applying feature standardization prior to model training.

Standardization ensures that all input features are on a comparable scale, which is particularly important when interpreting model coefficients and assessing the relative importance of macroeconomic variables.

The workflow includes:

- Scaling all input features using standardization
- Training a linear regression model on the transformed dataset
- Evaluating model performance relative to the non-standardized version
- Inspecting model coefficients to better understand feature impact

In [ ]:
# =========================================================
#  0. Limit the model to only the following features
# =========================================================

macro_features = [
    "unemployment_rate_lag_0",
    "gdp_growth_lag_1",
    "interest_rate_lag_0",
    "cpi_lag_0",
    "real_final_consumption_lag_7",
    "consumer_confidence_lag_1",
    "petrol_euro95_lag_4"
]

time_features = [
    "acea_total_registrations_lag_12",

]

In [ ]:
# =========================================================
#  1. Import StandardScaler
# =========================================================

from sklearn.preprocessing import StandardScaler

In [ ]:
# =========================================================
#  2. Define Features & Target
# =========================================================

FEATURES_V2 = time_features + macro_features

TARGET = "acea_total_registrations"

print("Features used:", FEATURES_V2)

In [ ]:
# =========================================================
#  3. Standardize Features
# =========================================================

scaler = StandardScaler()

X_scaled = scaler.fit_transform(macro_df[FEATURES_V2])

X_scaled_df = pd.DataFrame(X_scaled, columns=FEATURES_V2)

# Keep date + target
macro_scaled_df = pd.concat(
    [macro_df[["date", TARGET]].reset_index(drop=True), X_scaled_df],
    axis=1
)

In [ ]:
# =========================================================
#  Check Standardization (Mean & Std)
# =========================================================

check_df = macro_scaled_df[FEATURES_V2].agg(["mean", "std"]).T

display(check_df)

In [ ]:
# =========================================================
#  Quick Distribution Check
# =========================================================

macro_scaled_df[macro_features].hist(figsize=(12, 8))
plt.tight_layout()
plt.show()

In [ ]:
# =========================================================
#  4. Build Model
# =========================================================


from sklearn.linear_model import LinearRegression

model_v2 = LinearRegression()

In [ ]:
# =========================================================
#  5. Run Backtesting
# =========================================================

macro_v2_results = rolling_backtest(
    df=macro_scaled_df,
    start_date="2024-01-01",
    features=FEATURES_V2,
    target=TARGET,
    model=model_v2
)

In [ ]:
# =========================================================
#  6. Evaluate Macro Model v2
# =========================================================

from sklearn.metrics import mean_absolute_error, mean_squared_error
import numpy as np

macro_v2_mae = mean_absolute_error(
    macro_v2_results["actual"],
    macro_v2_results["prediction"]
)

macro_v2_rmse = np.sqrt(mean_squared_error(
    macro_v2_results["actual"],
    macro_v2_results["prediction"]
))

macro_v2_mape = np.mean(
    np.abs(
        (macro_v2_results["actual"] - macro_v2_results["prediction"]) 
        / macro_v2_results["actual"]
    )
) * 100

print(f"Macro v2 MAE: {macro_v2_mae:,.2f}")
print(f"Macro v2 RMSE: {macro_v2_rmse:,.2f}")
print(f"Macro v2 MAPE: {macro_v2_mape:.2f}%")

In [ ]:
# =========================================================
#  7. Compare Models
# =========================================================

baseline_mape = 7.59  # your earlier result

print(f"Baseline MAPE: {baseline_mape:.2f}%")
print(f"Macro v2 MAPE: {macro_v2_mape:.2f}%")
print(f"Macro  MAPE: {macro_mape:.2f}%")

In [ ]:
# =========================================================
#  Extract Model Coefficients
# =========================================================

coef_df_v2 = pd.DataFrame({
    "feature": FEATURES_V2,
    "coefficient": model_v2.coef_
})

coef_df_v2["abs_coef"] = coef_df_v2["coefficient"].abs()

# Sort by importance
coef_df_v2 = coef_df_v2.sort_values("abs_coef", ascending=False)

display(coef_df_v2)

In [ ]:
print("Intercept:", model_v2.intercept_)

In [ ]:
from sklearn.metrics import r2_score

# Fit model on full dataset (for interpretation only)
model_v2.fit(macro_scaled_df[FEATURES_V2], macro_scaled_df[TARGET])

y_pred_full = model_v2.predict(macro_scaled_df[FEATURES_V2])

r2 = r2_score(macro_scaled_df[TARGET], y_pred_full)

print(f"R² (in-sample): {r2:.4f}")

In [ ]:
# =========================================================
#  8. Visualize Results
# =========================================================

import matplotlib.pyplot as plt

plt.figure(figsize=(12,6))
plt.plot(macro_v2_results["date"], macro_v2_results["actual"], label="Actual")
plt.plot(macro_v2_results["date"], macro_v2_results["prediction"], label="Prediction")
plt.legend()
plt.title("Macro v2 Model - Rolling Forecast")
plt.show()

## Model 3 — Feature-Selected Macro Model

This model refines the previous macro model by selecting a subset of the most relevant features.

The objective is to improve model performance and interpretability by removing less informative variables and reducing potential noise introduced by weaker macroeconomic indicators.

The feature selection process is based on:

- model performance comparisons across different feature combinations
- coefficient stability and interpretability
- removal of variables with weak or inconsistent relationships

This results in a more parsimonious model that balances predictive accuracy and simplicity.

In [ ]:
# =========================================================
#  0. Limit the model to only the following features
# =========================================================

macro_features_2 = [
    "gdp_growth_lag_1",
    "interest_rate_lag_0",
    "cpi_lag_0",
    "real_final_consumption_lag_7",
    "petrol_euro95_lag_4"
]

time_features_2 = [
    "acea_total_registrations_lag_12",
]


# =========================================================
#  2. Define Features & Target
# =========================================================

FEATURES_V3 = time_features_2 + macro_features_2

TARGET = "acea_total_registrations"

print("Features used:", FEATURES_V3)
print("=========================================================")


# =========================================================
#  3. Standardize Features
# =========================================================

scaler = StandardScaler()

X_scaled = scaler.fit_transform(macro_df[FEATURES_V3])

X_scaled_df = pd.DataFrame(X_scaled, columns=FEATURES_V3)

# Keep date + target
macro_scaled_df_2 = pd.concat(
    [macro_df[["date", TARGET]].reset_index(drop=True), X_scaled_df],
    axis=1
)


# =========================================================
#  4. Build Model
# =========================================================

model_v3 = LinearRegression()

# =========================================================
#  5. Run Backtesting
# =========================================================

macro_v3_results = rolling_backtest(
    df=macro_scaled_df_2,
    start_date="2024-01-01",
    features=FEATURES_V3,
    target=TARGET,
    model=model_v3
)


# =========================================================
#  6. Evaluate Macro Model v3
# =========================================================

from sklearn.metrics import mean_absolute_error, mean_squared_error
import numpy as np

macro_v3_mae = mean_absolute_error(
    macro_v3_results["actual"],
    macro_v3_results["prediction"]
)

macro_v3_rmse = np.sqrt(mean_squared_error(
    macro_v3_results["actual"],
    macro_v3_results["prediction"]
))

macro_v3_mape = np.mean(
    np.abs(
        (macro_v3_results["actual"] - macro_v3_results["prediction"]) 
        / macro_v3_results["actual"]
    )
) * 100

print(f"Macro v3 MAE: {macro_v3_mae:,.2f}")
print(f"Macro v3 RMSE: {macro_v3_rmse:,.2f}")
print(f"Macro v3 MAPE: {macro_v3_mape:.2f}%")
print("=========================================================")

# =========================================================
#  7. Compare Models
# =========================================================

baseline_mape = 7.59  # your earlier result
print("Models Comparison: ")
print(f"Baseline MAPE: {baseline_mape:.2f}%")
print(f"Macro  MAPE: {macro_mape:.2f}%")
print(f"Macro v2 MAPE: {macro_v2_mape:.2f}%")
print(f"Macro v3 MAPE: {macro_v3_mape:.2f}%")



# =========================================================
#  8. Visualize Results
# =========================================================

import matplotlib.pyplot as plt

plt.figure(figsize=(12,6))
plt.plot(macro_v3_results["date"], macro_v3_results["actual"], label="Actual")
plt.plot(macro_v3_results["date"], macro_v3_results["prediction"], label="Prediction")
plt.legend()
plt.title("Macro v3 Model - Rolling Forecast")
plt.show()

# =========================================================
#  9. Create a Detailed Results Table
# =========================================================

macro_v3_results["error"] = macro_v3_results["prediction"] - macro_v3_results["actual"]
macro_v3_results["abs_error"] = np.abs(macro_v3_results["error"])
macro_v3_results["error_pct"] = (macro_v3_results["abs_error"] / macro_v3_results["actual"]) * 100

macro_v3_results_table = macro_v3_results[[
    "date",
    "actual",
    "prediction",
    "error",
    "abs_error",
    "error_pct"
]].copy()

macro_v3_results_table["actual"] = macro_v3_results_table["actual"].round(0)
macro_v3_results_table["prediction"] = macro_v3_results_table["prediction"].round(0)
macro_v3_results_table["error"] = macro_v3_results_table["error"].round(0)
macro_v3_results_table["abs_error"] = macro_v3_results_table["abs_error"].round(0)
macro_v3_results_table["error_pct"] = macro_v3_results_table["error_pct"].round(2)

print(macro_v3_results_table)

In [ ]:
# =========================================================
#  Extract Model Coefficients
# =========================================================

coef_df_v3 = pd.DataFrame({
    "feature": FEATURES_V3,
    "coefficient": model_v3.coef_
})

coef_df_v3["abs_coef"] = coef_df_v3["coefficient"].abs()

# Sort by importance
coef_df_v3 = coef_df_v3.sort_values("abs_coef", ascending=False)

display(coef_df_v3)


# =========================================================
#  Calculating Model R2
# =========================================================

from sklearn.metrics import r2_score

# Fit model on full dataset (for interpretation only)
model_v3.fit(macro_scaled_df_2[FEATURES_V3], macro_scaled_df_2[TARGET])

y_pred_full = model_v3.predict(macro_scaled_df_2[FEATURES_V3])

r2 = r2_score(macro_scaled_df_2[TARGET], y_pred_full)

print(f"R² (in-sample): {r2:.4f}")

## Model 4 — Linear Model with Enhanced Seasonality Features

This model builds on the previous feature-selected macro model by refining the representation of seasonality.

While earlier models capture seasonality using basic time-based features, this version introduces additional seasonal adjustments to better reflect recurring patterns in car registrations.

The objective is to improve the model’s ability to capture intra-year fluctuations, particularly peak and low-demand periods.

### Seasonality Enhancements

Seasonality is further captured through engineered features designed to reflect known market dynamics, such as peak registration months and seasonal dips.

This model represents the final linear regression specification before moving to regularized and non-linear models.

In [ ]:
# =========================================================
#  0. Create Monthly Seasonal Dummies
# =========================================================

macro_df_season = macro_df.copy()

macro_df_season["date"] = pd.to_datetime(macro_df_season["date"])
macro_df_season["month"] = macro_df_season["date"].dt.month

# Peak months: March and June
macro_df_season["is_peak_month"] = macro_df_season["month"].isin([3, 6]).astype(int)

# Dip month: Jan and August
macro_df_season["is_dip_month"] = macro_df_season["month"].isin([1, 8]).astype(int)


# =========================================================
#  1. Limit the model to only the following features
# =========================================================

macro_features_2 = [
    "gdp_growth_lag_1",
    "interest_rate_lag_0",
    "cpi_lag_0",
    "real_final_consumption_lag_7",
    "petrol_euro95_lag_4"
    
]

time_features_2 = [
    "acea_total_registrations_lag_12"
]

season_features = [
    "is_peak_month",
    "is_dip_month"
]


# =========================================================
#  2. Define Features & Target
# =========================================================

FEATURES_V4 = time_features_2 + macro_features_2 + season_features

TARGET = "acea_total_registrations"

print("Features used:", FEATURES_V4)
print("=========================================================")


# =========================================================
#  3. Standardize Features
# =========================================================

from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_absolute_error, mean_squared_error
import numpy as np
import pandas as pd

scaler = StandardScaler()

X_scaled = scaler.fit_transform(macro_df_season[FEATURES_V4])

X_scaled_df = pd.DataFrame(X_scaled, columns=FEATURES_V4)

# Keep date + target
macro_scaled_df_4 = pd.concat(
    [macro_df_season[["date", TARGET]].reset_index(drop=True), X_scaled_df],
    axis=1
)


# =========================================================
#  4. Build Model
# =========================================================

model_v4 = LinearRegression()


# =========================================================
#  5. Run Backtesting
# =========================================================

macro_v4_results = rolling_backtest(
    df=macro_scaled_df_4,
    start_date="2024-01-01",
    features=FEATURES_V4,
    target=TARGET,
    model=model_v4
)


# =========================================================
#  6. Evaluate Macro Model v4
# =========================================================

macro_v4_mae = mean_absolute_error(
    macro_v4_results["actual"],
    macro_v4_results["prediction"]
)

macro_v4_rmse = np.sqrt(mean_squared_error(
    macro_v4_results["actual"],
    macro_v4_results["prediction"]
))

macro_v4_mape = np.mean(
    np.abs(
        (macro_v4_results["actual"] - macro_v4_results["prediction"]) 
        / macro_v4_results["actual"]
    )
) * 100

print(f"Macro v4 MAE: {macro_v4_mae:,.2f}")
print(f"Macro v4 RMSE: {macro_v4_rmse:,.2f}")
print(f"Macro v4 MAPE: {macro_v4_mape:.2f}%")
print("=========================================================")


# =========================================================
#  7. Compare Models
# =========================================================

baseline_mape = 7.59

print("Models Comparison:")
print(f"Baseline MAPE: {baseline_mape:.2f}%")
print(f"Macro v1 MAPE: {macro_mape:.2f}%")
print(f"Macro v2 MAPE: {macro_v2_mape:.2f}%")
print(f"Macro v3 MAPE: {macro_v3_mape:.2f}%")
print(f"Macro v4 MAPE: {macro_v4_mape:.2f}%")


# =========================================================
#  8. Visualize Results
# =========================================================

import matplotlib.pyplot as plt

plt.figure(figsize=(12, 6))
plt.plot(macro_v4_results["date"], macro_v4_results["actual"], label="Actual")
plt.plot(macro_v4_results["date"], macro_v4_results["prediction"], label="Prediction")
plt.legend()
plt.title("Macro v4 Model - Rolling Forecast")
plt.show()


# =========================================================
#  9. Create a Detailed Results Table
# =========================================================

macro_v4_results["error"] = macro_v4_results["prediction"] - macro_v4_results["actual"]
macro_v4_results["abs_error"] = np.abs(macro_v4_results["error"])
macro_v4_results["error_pct"] = (
    macro_v4_results["abs_error"] / macro_v4_results["actual"]
) * 100

macro_v4_results_table = macro_v4_results[[
    "date",
    "actual",
    "prediction",
    "error",
    "abs_error",
    "error_pct"
]].copy()

macro_v4_results_table["actual"] = macro_v4_results_table["actual"].round(0)
macro_v4_results_table["prediction"] = macro_v4_results_table["prediction"].round(0)
macro_v4_results_table["error"] = macro_v4_results_table["error"].round(0)
macro_v4_results_table["abs_error"] = macro_v4_results_table["abs_error"].round(0)
macro_v4_results_table["error_pct"] = macro_v4_results_table["error_pct"].round(2)

print(macro_v4_results_table)

# =========================================================
#  10. Extract Model Coefficients
# =========================================================

coef_df_v4 = pd.DataFrame({
    "feature": FEATURES_V4,
    "coefficient": model_v4.coef_
})

coef_df_v4["abs_coef"] = coef_df_v4["coefficient"].abs()

# Sort by importance
coef_df_v4 = coef_df_v4.sort_values("abs_coef", ascending=False)

display(coef_df_v4)
print("=========================================================")


# =========================================================
#  11. Calculating Model R2
# =========================================================

from sklearn.metrics import r2_score

# Fit model on full dataset (for interpretation only)
model_v4.fit(macro_scaled_df_4[FEATURES_V4], macro_scaled_df_4[TARGET])

y_pred_full = model_v4.predict(macro_scaled_df_4[FEATURES_V4])

r2 = r2_score(macro_scaled_df_4[TARGET], y_pred_full)

print(f"R² (in-sample): {r2:.4f}")

# Developing a Ridge based macro model  

This model introduces regularization through Ridge regression to improve model stability and generalization.

While previous linear models may be sensitive to multicollinearity between macroeconomic variables, Ridge regression applies L2 regularization to reduce coefficient volatility and prevent overfitting.

The objective is to:

- stabilize coefficient estimates
- improve predictive performance
- handle correlated macroeconomic features more effectively

This model serves as the first regularized alternative to the standard linear regression models.

In [ ]:
# =========================================================
#  1. Import Ridge
# =========================================================

from sklearn.linear_model import Ridge

# =========================================================
#  2. Define Alpha Grid
# =========================================================

alphas = [0.01, 0.1, 1, 10, 50, 100]


In [ ]:
# =========================================================
#  3. Tune Ridge Alpha using Backtesting
# =========================================================

ridge_results_summary = []

for alpha in alphas:
    
    model_ridge = Ridge(alpha=alpha)
    
    res = rolling_backtest(
        df=macro_scaled_df_4,
        start_date="2024-01-01",
        features=FEATURES_V4,
        target=TARGET,
        model=model_ridge
    )
    
    mape = np.mean(
        np.abs((res["actual"] - res["prediction"]) / res["actual"])
    ) * 100
    
    ridge_results_summary.append({
        "alpha": alpha,
        "MAPE": round(mape, 3)
    })

ridge_summary_df = pd.DataFrame(ridge_results_summary).sort_values("MAPE")

print(ridge_summary_df)

In [ ]:
best_alpha = ridge_summary_df.iloc[0]["alpha"]

model_ridge_v1 = Ridge(alpha=best_alpha)

ridge_v1_results = rolling_backtest(
    df=macro_scaled_df_4,
    start_date="2024-01-01",
    features=FEATURES_V4,
    target=TARGET,
    model=model_ridge_v1
)

In [ ]:
# =========================================================
#  6. Evaluate Model Performance
# =========================================================

from sklearn.metrics import mean_absolute_error, mean_squared_error

ridge_v1_mae = mean_absolute_error(
    ridge_v1_results["actual"],
    ridge_v1_results["prediction"]
)

ridge_v1_rmse = np.sqrt(mean_squared_error(
    ridge_v1_results["actual"],
    ridge_v1_results["prediction"]
))

ridge_v1_mape = np.mean(
    np.abs(
        (ridge_v1_results["actual"] - ridge_v1_results["prediction"]) 
        / ridge_v1_results["actual"]
    )
) * 100

print(f"Ridge v5 MAE: {ridge_v1_mae:,.2f}")
print(f"Ridge v5 RMSE: {ridge_v1_rmse:,.2f}")
print(f"Ridge v5 MAPE: {ridge_v1_mape:.2f}%")
print("=========================================================")


# =========================================================
#  7. Compare Models
# =========================================================

baseline_mape = 7.59

print("Models Comparison:")
print(f"Baseline MAPE: {baseline_mape:.2f}%")
print(f"Macro v1 MAPE: {macro_mape:.2f}%")
print(f"Macro v2 MAPE: {macro_v2_mape:.2f}%")
print(f"Macro v3 MAPE: {macro_v3_mape:.2f}%")
print(f"Macro v4 MAPE: {macro_v4_mape:.2f}%")
print(f"Ridge v1 MAPE: {ridge_v1_mape:.2f}%")



# =========================================================
#  8. Visualize Results
# =========================================================

import matplotlib.pyplot as plt

plt.figure(figsize=(12, 6))
plt.plot(ridge_v1_results["date"], ridge_v1_results["actual"], label="Actual")
plt.plot(ridge_v1_results["date"], ridge_v1_results["prediction"], label="Prediction")
plt.legend()
plt.title("Ridge Model - Rolling Forecast")
plt.show()


# =========================================================
#  9. Creating Detailed results table
# =========================================================


ridge_v1_results["error"] = ridge_v1_results ["prediction"] - ridge_v1_results["actual"]
ridge_v1_results["abs_error"] = np.abs(ridge_v1_results["error"])
ridge_v1_results["error_pct"] = (
    ridge_v1_results["abs_error"] / ridge_v1_results["actual"]
) * 100

ridge_v1_results = ridge_v1_results[[
    "date",
    "actual",
    "prediction",
    "error",
    "abs_error",
    "error_pct"
]].copy()

ridge_v1_results["actual"] = ridge_v1_results["actual"].round(0)
ridge_v1_results["prediction"] = ridge_v1_results["prediction"].round(0)
ridge_v1_results["error"] = ridge_v1_results["error"].round(0)
ridge_v1_results["abs_error"] = ridge_v1_results["abs_error"].round(0)
ridge_v1_results["error_pct"] = ridge_v1_results["error_pct"].round(2)

print(ridge_v1_results)


# =========================================================
#  10. Extract Model Coefficients
# =========================================================

coef_df_v5 = pd.DataFrame({
    "feature": FEATURES_V4,
    "coefficient": model_ridge_v1.coef_
})

coef_df_v5["abs_coef"] = coef_df_v5["coefficient"].abs()

# Sort by importance
coef_df_v5 = coef_df_v5.sort_values("abs_coef", ascending=False)

display(coef_df_v5)
print("=========================================================")


# =========================================================
#  11. Calculating Model R2
# =========================================================

from sklearn.metrics import r2_score

# Fit model on full dataset (for interpretation only)
model_ridge_v1.fit(macro_scaled_df_4[FEATURES_V4], macro_scaled_df_4[TARGET])

y_pred_full = model_ridge_v1.predict(macro_scaled_df_4[FEATURES_V4])

r2 = r2_score(macro_scaled_df_4[TARGET], y_pred_full)

print(f"R² (in-sample): {r2:.4f}")

## Model 6 — Ridge Regression with Industry Variables

This model extends the Ridge regression framework by incorporating additional industry-level indicators.

While previous models focused on macroeconomic drivers, this version evaluates whether industry-specific variables (such as vehicle parc and production) provide additional predictive power.

The objective is to:

- capture structural dynamics of the automotive market
- complement macroeconomic signals with industry context
- assess the incremental value of industry features in forecasting performance

This model helps determine whether combining macroeconomic and industry indicators improves forecast accuracy.

In [ ]:
# =========================================================
# Load datasets
# =========================================================
full_model_df = pd.read_parquet("data/full_model_df.parquet")

In [ ]:
# =========================================================
#  0. Prepare Dataset (Add vehicle_production)
# =========================================================

macro_df_ind = full_model_df.copy()  # use dataset with seasonal dummies already created

# =========================================================
#  0. Create Monthly Seasonal Dummies
# =========================================================

macro_df_ind["date"] = pd.to_datetime(macro_df_ind["date"])
macro_df_ind["month"] = macro_df_ind["date"].dt.month

# Peak months: March and June
macro_df_ind["is_peak_month"] = macro_df_ind["month"].isin([3, 6]).astype(int)

# Dip month: Jan and August
macro_df_ind["is_dip_month"] = macro_df_ind["month"].isin([1, 8]).astype(int)


# =========================================================
#  1. Limit the model to only the following features
# =========================================================

macro_features_3 = [
    "gdp_growth_lag_1",
    "interest_rate_lag_0",
    "cpi_lag_0",
    "real_final_consumption_lag_7",
    "petrol_euro95_lag_4",
    "vehicle_production_lag_1",
    
]

time_features_3 = [
]

season_features = [
    "is_peak_month",
    "is_dip_month"
]


# =========================================================
#  2. Define Features & Target
# =========================================================

FEATURES_V5 = time_features_3 + macro_features_3 + season_features

TARGET = "acea_total_registrations"


print("Features used:", FEATURES_V5)
print("=========================================================")


# =========================================================
#  1. Import Ridge
# =========================================================

from sklearn.linear_model import Ridge


# =========================================================
#  2. Define Alpha Grid
# =========================================================

alphas = [0.01, 0.1, 1, 10, 50, 100]


# =========================================================
#  3. Standardize Features
# =========================================================

from sklearn.preprocessing import StandardScaler

scaler = StandardScaler()

X_scaled = scaler.fit_transform(macro_df_ind[FEATURES_V5])

X_scaled_df = pd.DataFrame(X_scaled, columns=FEATURES_V5)

macro_scaled_df_5 = pd.concat(
    [macro_df_ind[["date", TARGET]].reset_index(drop=True), X_scaled_df],
    axis=1
)


# =========================================================
#  4. Tune Ridge Alpha using Backtesting
# =========================================================

import numpy as np
import pandas as pd

ridge_results_summary = []

for alpha in alphas:
    
    model_ridge = Ridge(alpha=alpha)
    
    res = rolling_backtest(
        df=macro_scaled_df_5,
        start_date="2024-01-01",
        features=FEATURES_V5,
        target=TARGET,
        model=model_ridge
    )
    
    mape = np.mean(
        np.abs((res["actual"] - res["prediction"]) / res["actual"])
    ) * 100
    
    ridge_results_summary.append({
        "alpha": alpha,
        "MAPE": round(mape, 3)
    })

ridge_summary_df = pd.DataFrame(ridge_results_summary).sort_values("MAPE")

print(ridge_summary_df)


# =========================================================
#  5. Choose Best Alpha
# =========================================================

best_alpha = ridge_summary_df.iloc[0]["alpha"]

print(f"Best alpha selected: {best_alpha}")


# =========================================================
#  6. Run Final Ridge Model with Best Alpha
# =========================================================

model_ridge_final = Ridge(alpha=best_alpha)

ridge_v5_results = rolling_backtest(
    df=macro_scaled_df_5,
    start_date="2024-01-01",
    features=FEATURES_V5,
    target=TARGET,
    model=model_ridge_final
)


# =========================================================
#  7. Evaluate Model Performance
# =========================================================

from sklearn.metrics import mean_absolute_error, mean_squared_error

ridge_v5_mae = mean_absolute_error(
    ridge_v5_results["actual"],
    ridge_v5_results["prediction"]
)

ridge_v5_rmse = np.sqrt(mean_squared_error(
    ridge_v5_results["actual"],
    ridge_v5_results["prediction"]
))

ridge_v5_mape = np.mean(
    np.abs(
        (ridge_v5_results["actual"] - ridge_v5_results["prediction"]) 
        / ridge_v5_results["actual"]
    )
) * 100

print(f"Ridge v5 MAE: {ridge_v5_mae:,.2f}")
print(f"Ridge v5 RMSE: {ridge_v5_rmse:,.2f}")
print(f"Ridge v5 MAPE: {ridge_v5_mape:.2f}%")
print("=========================================================")


# =========================================================
#  8. Compare Models
# =========================================================

print("Models Comparison:")
print(f"Baseline MAPE: {baseline_mape:.2f}%")
print(f"Macro v3 MAPE: {macro_v3_mape:.2f}%")
print(f"Macro v4 MAPE: {macro_v4_mape:.2f}%")
print(f"Ridge v1 MAPE: {ridge_v1_mape:.2f}%")
print(f"Ridge v5 (+vehicle_production) MAPE: {ridge_v5_mape:.2f}%")
print("=========================================================")


# =========================================================
#  9. Visualize Results
# =========================================================

import matplotlib.pyplot as plt

plt.figure(figsize=(12,6))
plt.plot(ridge_v5_results["date"], ridge_v5_results["actual"], label="Actual")
plt.plot(ridge_v5_results["date"], ridge_v5_results["prediction"], label="Prediction")
plt.legend()
plt.title("Ridge v5 Model - Rolling Forecast (+vehicle_production)")
plt.show()


# =========================================================
#  10. Create Detailed Results Table
# =========================================================

ridge_v5_results["error"] = ridge_v5_results["prediction"] - ridge_v5_results["actual"]
ridge_v5_results["abs_error"] = np.abs(ridge_v5_results["error"])
ridge_v5_results["error_pct"] = (
    ridge_v5_results["abs_error"] / ridge_v5_results["actual"]
) * 100

ridge_v5_table = ridge_v5_results[[
    "date",
    "actual",
    "prediction",
    "error",
    "abs_error",
    "error_pct"
]].copy()

ridge_v5_table["actual"] = ridge_v5_table["actual"].round(0)
ridge_v5_table["prediction"] = ridge_v5_table["prediction"].round(0)
ridge_v5_table["error"] = ridge_v5_table["error"].round(0)
ridge_v5_table["abs_error"] = ridge_v5_table["abs_error"].round(0)
ridge_v5_table["error_pct"] = ridge_v5_table["error_pct"].round(2)

print(ridge_v5_table)



In [ ]:

# =========================================================
#  11. Extract Model Coefficients
# =========================================================

model_ridge_final.fit(
    macro_scaled_df_5[FEATURES_V5],
    macro_scaled_df_5[TARGET]
)

ridge_coef_df = pd.DataFrame({
    "feature": FEATURES_V5,
    "coefficient": model_ridge_final.coef_
})

ridge_coef_df["abs_coef"] = ridge_coef_df["coefficient"].abs()
ridge_coef_df = ridge_coef_df.sort_values("abs_coef", ascending=False)

print(ridge_coef_df)
print("Intercept:", model_ridge_final.intercept_)


# =========================================================
#  12. Calculate R²
# =========================================================

r2 = model_ridge_final.score(
    macro_scaled_df_5[FEATURES_V5],
    macro_scaled_df_5[TARGET]
)

print(f"R² (in-sample): {r2:.4f}")

## Model 7 — Elastic Net Regression Model

This model introduces Elastic Net regression as an extension of the Ridge-based approach.

Elastic Net combines both L1 (Lasso) and L2 (Ridge) regularization, allowing the model to both shrink coefficients and perform implicit feature selection.

The objective is to:

- balance coefficient stability (Ridge) with sparsity (Lasso)
- reduce the impact of multicollinearity
- potentially eliminate less relevant features automatically

The model is trained using the same feature set as the previous Ridge model to allow for a direct comparison of performance.

In [ ]:
# =========================================================
#  1. Import ElasticNet
# =========================================================

from sklearn.linear_model import ElasticNet


# =========================================================
#  2. Define Grid
# =========================================================

alphas = [0.01, 0.1, 1, 10]
l1_ratios = [0.1, 0.3, 0.5, 0.7]


# =========================================================
#  3. Tune ElasticNet
# =========================================================

elastic_results = []

for alpha in alphas:
    for l1 in l1_ratios:
        
        model = ElasticNet(alpha=alpha, l1_ratio=l1, max_iter=10000)
        
        res = rolling_backtest(
            df=macro_scaled_df_5,   # same dataset as Ridge v4
            start_date="2024-01-01",
            features=FEATURES_V5,
            target=TARGET,
            model=model
        )
        
        mape = np.mean(
            np.abs((res["actual"] - res["prediction"]) / res["actual"])
        ) * 100
        
        elastic_results.append({
            "alpha": alpha,
            "l1_ratio": l1,
            "MAPE": round(mape, 3)
        })

elastic_summary = pd.DataFrame(elastic_results).sort_values("MAPE")

print(elastic_summary)

## Model 8 — Random Forest Regression

This model introduces a non-linear approach using Random Forest regression.

While previous models rely on linear relationships between features and the target variable, Random Forest can capture more complex patterns and interactions between variables.

The objective is to:

- model non-linear relationships in the data
- capture interactions between macroeconomic and industry variables
- improve predictive performance beyond linear models

Random Forest is an ensemble method that combines multiple decision trees, helping to reduce overfitting while maintaining flexibility in capturing complex dynamics.

In [ ]:
print(macro_df_ind.columns.tolist())

In [ ]:
# =========================================================
#  0. Prepare Dataset for Random Forest
# =========================================================
# What this step does:
# Creates a clean dataset with the same best-performing feature set,
# but without standardization, since Random Forest does not need scaling.

rf_df = macro_df_ind.copy()

TARGET = "acea_total_registrations"

FEATURES_RF = [
    "acea_total_registrations_lag_12",
    "gdp_growth_lag_1",
    "interest_rate_lag_0",
    "cpi_lag_0",
    "real_final_consumption_lag_7",
    "petrol_euro95_lag_4",
    "is_peak_month",
    "is_dip_month"
    
]

rf_df = rf_df[["date", TARGET] + FEATURES_RF].dropna().reset_index(drop=True)

print("Features used:", FEATURES_RF)
print("=========================================================")


# =========================================================
#  1. Import Random Forest
# =========================================================
# What this step does:
# Imports the Random Forest regressor.

from sklearn.ensemble import RandomForestRegressor


# =========================================================
#  2. Define Hyperparameter Grid
# =========================================================
# What this step does:
# Creates a small grid of parameter combinations to test.
# We keep it reasonably small because rolling backtesting is computationally expensive.

rf_param_grid = [
    {"n_estimators": 100, "max_depth": 3, "min_samples_leaf": 1},
    {"n_estimators": 200, "max_depth": 3, "min_samples_leaf": 1},
    {"n_estimators": 100, "max_depth": 5, "min_samples_leaf": 1},
    {"n_estimators": 200, "max_depth": 5, "min_samples_leaf": 1},
    {"n_estimators": 100, "max_depth": 5, "min_samples_leaf": 2},
    {"n_estimators": 200, "max_depth": 5, "min_samples_leaf": 2},
    {"n_estimators": 100, "max_depth": 7, "min_samples_leaf": 1},
    {"n_estimators": 200, "max_depth": 7, "min_samples_leaf": 2},
]


# =========================================================
#  3. Tune Random Forest using Backtesting
# =========================================================
# What this step does:
# Tests multiple Random Forest settings using your rolling backtest
# and compares them using MAPE.

import numpy as np
import pandas as pd

rf_results_summary = []

for params in rf_param_grid:
    
    model_rf = RandomForestRegressor(
        n_estimators=params["n_estimators"],
        max_depth=params["max_depth"],
        min_samples_leaf=params["min_samples_leaf"],
        random_state=42
    )
    
    res = rolling_backtest(
        df=rf_df,
        start_date="2024-01-01",
        features=FEATURES_RF,
        target=TARGET,
        model=model_rf
    )
    
    mape = np.mean(
        np.abs((res["actual"] - res["prediction"]) / res["actual"])
    ) * 100
    
    rf_results_summary.append({
        "n_estimators": params["n_estimators"],
        "max_depth": params["max_depth"],
        "min_samples_leaf": params["min_samples_leaf"],
        "MAPE": round(mape, 3)
    })

rf_summary_df = pd.DataFrame(rf_results_summary).sort_values("MAPE")

print(rf_summary_df)

In [ ]:
# =========================================================
#  4. Choose Best Hyperparameters
# =========================================================
# What this step does:
# Selects the best-performing Random Forest setup based on lowest MAPE.

best_rf_params = rf_summary_df.iloc[0]

best_n_estimators = int(best_rf_params["n_estimators"])
best_max_depth = int(best_rf_params["max_depth"])
best_min_samples_leaf = int(best_rf_params["min_samples_leaf"])

print("Best Random Forest parameters selected:")
print(f"n_estimators = {best_n_estimators}")
print(f"max_depth = {best_max_depth}")
print(f"min_samples_leaf = {best_min_samples_leaf}")
print("=========================================================")


# =========================================================
#  5. Run Final Random Forest Model
# =========================================================
# What this step does:
# Re-runs the model using the best parameter combination found above.

model_rf_final = RandomForestRegressor(
    n_estimators=best_n_estimators,
    max_depth=best_max_depth,
    min_samples_leaf=best_min_samples_leaf,
    random_state=42
)

rf_final_results = rolling_backtest(
    df=rf_df,
    start_date="2024-01-01",
    features=FEATURES_RF,
    target=TARGET,
    model=model_rf_final
)

# =========================================================
#  6. Evaluate Model Performance
# =========================================================
# What this step does:
# Calculates MAE, RMSE, and MAPE for the final Random Forest model.

from sklearn.metrics import mean_absolute_error, mean_squared_error

rf_final_mae = mean_absolute_error(
    rf_final_results["actual"],
    rf_final_results["prediction"]
)

rf_final_rmse = np.sqrt(mean_squared_error(
    rf_final_results["actual"],
    rf_final_results["prediction"]
))

rf_final_mape = np.mean(
    np.abs(
        (rf_final_results["actual"] - rf_final_results["prediction"]) 
        / rf_final_results["actual"]
    )
) * 100

print(f"Random Forest MAE: {rf_final_mae:,.2f}")
print(f"Random Forest RMSE: {rf_final_rmse:,.2f}")
print(f"Random Forest MAPE: {rf_final_mape:.2f}%")
print("=========================================================")

In [ ]:
# =========================================================
#  8. Visualize Results
# =========================================================
# What this step does:
# Plots actual vs predicted values over time.

import matplotlib.pyplot as plt

plt.figure(figsize=(12, 6))
plt.plot(rf_final_results["date"], rf_final_results["actual"], label="Actual")
plt.plot(rf_final_results["date"], rf_final_results["prediction"], label="Prediction")
plt.legend()
plt.title("Random Forest Model - Rolling Forecast")
plt.show()

In [ ]:
# =========================================================
#  9. Create Detailed Results Table
# =========================================================
# What this step does:
# Builds a month-level table with actuals, predictions, and error metrics.

rf_final_results["error"] = rf_final_results["prediction"] - rf_final_results["actual"]
rf_final_results["abs_error"] = np.abs(rf_final_results["error"])
rf_final_results["error_pct"] = (
    rf_final_results["abs_error"] / rf_final_results["actual"]
) * 100

rf_results_table = rf_final_results[[
    "date",
    "actual",
    "prediction",
    "error",
    "abs_error",
    "error_pct"
]].copy()

rf_results_table["actual"] = rf_results_table["actual"].round(0)
rf_results_table["prediction"] = rf_results_table["prediction"].round(0)
rf_results_table["error"] = rf_results_table["error"].round(0)
rf_results_table["abs_error"] = rf_results_table["abs_error"].round(0)
rf_results_table["error_pct"] = rf_results_table["error_pct"].round(2)

print(rf_results_table)

In [ ]:
# =========================================================
#  10. Extract Feature Importance
# =========================================================
# What this step does:
# Random Forest does not give regression coefficients like linear models,
# but it does provide feature importance scores.

model_rf_final.fit(rf_df[FEATURES_RF], rf_df[TARGET])

rf_importance_df = pd.DataFrame({
    "feature": FEATURES_RF,
    "importance": model_rf_final.feature_importances_
})

rf_importance_df = rf_importance_df.sort_values("importance", ascending=False)

print(rf_importance_df)

# =========================================================
#  11. Calculate R²
# =========================================================
# What this step does:
# Calculates in-sample R² for the final Random Forest model.

rf_r2 = model_rf_final.score(
    rf_df[FEATURES_RF],
    rf_df[TARGET]
)

print(f"R² (in-sample): {rf_r2:.4f}")

## Model 9 — XGBoost Regression Model

This model applies Extreme Gradient Boosting (XGBoost), a powerful ensemble learning technique designed to capture complex non-linear relationships in the data.

XGBoost builds sequential decision trees, where each new tree corrects the errors of the previous ones. This allows the model to learn intricate patterns and interactions between features more effectively than traditional methods.

The objective is to:

- improve predictive performance beyond Random Forest
- capture non-linear relationships and feature interactions
- provide a strong benchmark for comparison with other models

This model represents the most advanced modeling approach tested in this workflow.

In [ ]:
from xgboost import XGBRegressor

In [ ]:
# =========================================================
#  0. Prepare Dataset
# =========================================================

# Use SAME dataset as Random Forest (no scaling)
xgb_df = macro_df_ind.copy()

TARGET = "acea_total_registrations"

FEATURES_XGB = [
    "acea_total_registrations_lag_12",
    "gdp_growth_lag_1",
    "interest_rate_lag_0",
    "cpi_lag_0",
    "real_final_consumption_lag_7",
    "petrol_euro95_lag_4",
    "is_peak_month",
    "is_dip_month",
]

xgb_df = xgb_df[["date", TARGET] + FEATURES_XGB].dropna().reset_index(drop=True)

print("Features used:", FEATURES_XGB)
print("=========================================================")

# =========================================================
#  2. Define Parameter Grid
# =========================================================

xgb_param_grid = [
    {"n_estimators": 100, "max_depth": 2, "learning_rate": 0.05},
    {"n_estimators": 200, "max_depth": 2, "learning_rate": 0.05},
    {"n_estimators": 100, "max_depth": 3, "learning_rate": 0.05},
    {"n_estimators": 200, "max_depth": 3, "learning_rate": 0.05},
    {"n_estimators": 100, "max_depth": 3, "learning_rate": 0.1},
    {"n_estimators": 200, "max_depth": 3, "learning_rate": 0.1},
    {"n_estimators": 100, "max_depth": 4, "learning_rate": 0.05},
    {"n_estimators": 200, "max_depth": 4, "learning_rate": 0.05},
]

# =========================================================
#  3. Tune XGBoost using Backtesting
# =========================================================

import numpy as np
import pandas as pd

xgb_results_summary = []

for params in xgb_param_grid:
    
    model_xgb = XGBRegressor(
        n_estimators=params["n_estimators"],
        max_depth=params["max_depth"],
        learning_rate=params["learning_rate"],
        random_state=42,
        objective="reg:squarederror"
    )
    
    res = rolling_backtest(
        df=xgb_df,
        start_date="2024-01-01",
        features=FEATURES_XGB,
        target=TARGET,
        model=model_xgb
    )
    
    mape = np.mean(
        np.abs((res["actual"] - res["prediction"]) / res["actual"])
    ) * 100
    
    xgb_results_summary.append({
        "n_estimators": params["n_estimators"],
        "max_depth": params["max_depth"],
        "learning_rate": params["learning_rate"],
        "MAPE": round(mape, 3)
    })

xgb_summary_df = pd.DataFrame(xgb_results_summary).sort_values("MAPE")

print(xgb_summary_df)

# =========================================================
#  4. Select Best Parameters
# =========================================================

best_params = xgb_summary_df.iloc[0]

best_n = int(best_params["n_estimators"])
best_depth = int(best_params["max_depth"])
best_lr = best_params["learning_rate"]

print("Best parameters:")
print(best_params)
print("=========================================================")

# =========================================================
#  5. Final Model
# =========================================================

model_xgb_final = XGBRegressor(
    n_estimators=best_n,
    max_depth=best_depth,
    learning_rate=best_lr,
    random_state=42,
    objective="reg:squarederror"
)

xgb_final_results = rolling_backtest(
    df=xgb_df,
    start_date="2024-01-01",
    features=FEATURES_XGB,
    target=TARGET,
    model=model_xgb_final
)


# =========================================================
#  6. Evaluate Performance
# =========================================================

from sklearn.metrics import mean_absolute_error, mean_squared_error

xgb_mae = mean_absolute_error(
    xgb_final_results["actual"],
    xgb_final_results["prediction"]
)

xgb_rmse = np.sqrt(mean_squared_error(
    xgb_final_results["actual"],
    xgb_final_results["prediction"]
))

xgb_mape = np.mean(
    np.abs(
        (xgb_final_results["actual"] - xgb_final_results["prediction"]) 
        / xgb_final_results["actual"]
    )
) * 100

print(f"XGBoost MAE: {xgb_mae:,.2f}")
print(f"XGBoost RMSE: {xgb_rmse:,.2f}")
print(f"XGBoost MAPE: {xgb_mape:.2f}%")
print("=========================================================")

# =========================================================
#  7. Compare Models
# =========================================================

print("Models Comparison:")
print(f"Baseline MAPE: {baseline_mape:.2f}%")
print(f"Macro v3 MAPE: {macro_v3_mape:.2f}%")
print(f"Macro v4 MAPE: {macro_v4_mape:.2f}%")
print(f"ridge_v1_mape: {ridge_v1_mape:.2f}%")
print(f"ridge_v5_mape: {ridge_v5_mape:.2f}%")
print(f"Random Forest MAPE: 5.34%")
print(f"XGBoost MAPE: {xgb_mape:.2f}%")
print("=========================================================")


In [ ]:
# =========================================================
#  8. Feature Importance
# =========================================================

model_xgb_final.fit(xgb_df[FEATURES_XGB], xgb_df[TARGET])

importance_df = pd.DataFrame({
    "feature": FEATURES_XGB,
    "importance": model_xgb_final.feature_importances_
}).sort_values("importance", ascending=False)

print(importance_df)

In [ ]:
# =========================================================
#  9. Detailed Results Table
# =========================================================

xgb_final_results["error"] = xgb_final_results["prediction"] - xgb_final_results["actual"]
xgb_final_results["abs_error"] = np.abs(xgb_final_results["error"])
xgb_final_results["error_pct"] = (
    xgb_final_results["abs_error"] / xgb_final_results["actual"]
) * 100

print(xgb_final_results)



In [ ]:

# =========================================================
#  11. Visualize Results
# =========================================================
# What this step does:
# Plots actual vs predicted values over time.

import matplotlib.pyplot as plt

plt.figure(figsize=(12, 6))
plt.plot(xgb_final_results["date"], xgb_final_results["actual"], label="Actual")
plt.plot(xgb_final_results["date"], xgb_final_results["prediction"], label="Prediction")
plt.legend()
plt.title("XGBoost - Rolling Forecast")
plt.show()

## Model 10 — XGBoost with Industry Variables

This model extends the XGBoost framework by incorporating industry-level indicators alongside macroeconomic variables.

While the previous XGBoost model focused on macroeconomic drivers, this version evaluates whether adding structural industry signals (such as vehicle parc and production) improves predictive performance.

The objective is to:

- capture both macroeconomic and industry-specific dynamics
- enhance the model’s ability to explain variations in car registrations
- assess the incremental value of industry features within a non-linear modeling framework

This model represents the most comprehensive feature set tested in the modeling process.

In [ ]:
print(macro_df_ind.columns.tolist())

In [ ]:
# =========================================================
#  0. Prepare Dataset
# =========================================================

# Use SAME dataset as Random Forest (no scaling)
xgb_df_v2 = macro_df_ind.copy()

TARGET = "acea_total_registrations"

FEATURES_XGB = [
    "vehicle_parc_lag_0",
    "vehicle_production_lag_1",
    "interest_rate_lag_0",
    "cpi_lag_0",
    "real_final_consumption_lag_7",
    "petrol_euro95_lag_4",
    "gdp_growth_lag_1",
    "is_peak_month",
    "is_dip_month"
]

xgb_df_v2 = xgb_df_v2[["date", TARGET] + FEATURES_XGB].dropna().reset_index(drop=True)

print("Features used:", FEATURES_XGB)
print("=========================================================")

# =========================================================
#  2. Define Parameter Grid
# =========================================================

xgb_param_grid = [
    {"n_estimators": 100, "max_depth": 2, "learning_rate": 0.05},
    {"n_estimators": 200, "max_depth": 2, "learning_rate": 0.05},
    {"n_estimators": 100, "max_depth": 3, "learning_rate": 0.05},
    {"n_estimators": 200, "max_depth": 3, "learning_rate": 0.05},
    {"n_estimators": 100, "max_depth": 3, "learning_rate": 0.1},
    {"n_estimators": 200, "max_depth": 3, "learning_rate": 0.1},
    {"n_estimators": 100, "max_depth": 4, "learning_rate": 0.05},
    {"n_estimators": 200, "max_depth": 4, "learning_rate": 0.05},
]

# =========================================================
#  3. Tune XGBoost using Backtesting
# =========================================================

import numpy as np
import pandas as pd

xgb_results_summary = []

for params in xgb_param_grid:
    
    model_xgb_v2 = XGBRegressor(
        n_estimators=params["n_estimators"],
        max_depth=params["max_depth"],
        learning_rate=params["learning_rate"],
        random_state=42,
        objective="reg:squarederror"
    )
    
    res = rolling_backtest(
        df=xgb_df_v2,
        start_date="2024-01-01",
        features=FEATURES_XGB,
        target=TARGET,
        model=model_xgb_v2
    )
    
    mape = np.mean(
        np.abs((res["actual"] - res["prediction"]) / res["actual"])
    ) * 100
    
    xgb_results_summary.append({
        "n_estimators": params["n_estimators"],
        "max_depth": params["max_depth"],
        "learning_rate": params["learning_rate"],
        "MAPE": round(mape, 3)
    })

xgb_summary_df = pd.DataFrame(xgb_results_summary).sort_values("MAPE")

print(xgb_summary_df)

# =========================================================
#  4. Select Best Parameters
# =========================================================

best_params = xgb_summary_df.iloc[0]

best_n = int(best_params["n_estimators"])
best_depth = int(best_params["max_depth"])
best_lr = best_params["learning_rate"]

print("Best parameters:")
print(best_params)
print("=========================================================")

# =========================================================
#  5. Final Model
# =========================================================

model_xgb_v2_final = XGBRegressor(
    n_estimators=best_n,
    max_depth=best_depth,
    learning_rate=best_lr,
    random_state=42,
    objective="reg:squarederror"
)

xgb_v2_final_results = rolling_backtest(
    df=xgb_df_v2,
    start_date="2024-01-01",
    features=FEATURES_XGB,
    target=TARGET,
    model=model_xgb_v2_final
)


# =========================================================
#  6. Evaluate Performance
# =========================================================

from sklearn.metrics import mean_absolute_error, mean_squared_error

xgb_v2_mae = mean_absolute_error(
    xgb_v2_final_results["actual"],
    xgb_v2_final_results["prediction"]
)

xgb_v2_rmse = np.sqrt(mean_squared_error(
    xgb_v2_final_results["actual"],
    xgb_v2_final_results["prediction"]
))

xgb_v2_mape = np.mean(
    np.abs(
        (xgb_v2_final_results["actual"] - xgb_v2_final_results["prediction"]) 
        / xgb_v2_final_results["actual"]
    )
) * 100

print(f"XGBoost v2 MAE: {xgb_v2_mae:,.2f}")
print(f"XGBoost v2 RMSE: {xgb_v2_rmse:,.2f}")
print(f"XGBoost v2 MAPE: {xgb_v2_mape:.2f}%")
print("=========================================================")

# =========================================================
#  7. Compare Models
# =========================================================

print("Models Comparison:")
print(f"Baseline MAPE: {baseline_mape:.2f}%")
print(f"Macro v3 MAPE: {macro_v3_mape:.2f}%")
print(f"Macro v4 MAPE: {macro_v4_mape:.2f}%")
print(f"ridge_v1_mape: {ridge_v1_mape:.2f}%")
print(f"ridge_v5_mape: {ridge_v5_mape:.2f}%")
print(f"Random Forest MAPE: 5.34%")
print(f"XGBoost_v1 MAPE: {xgb_mape:.2f}%")
print(f"XGBoost_v2 MAPE: {xgb_v2_mape:.2f}%")
print("=========================================================")


# =========================================================
#  8. Feature Importance
# =========================================================

model_xgb_v2_final.fit(xgb_df_v2[FEATURES_XGB], xgb_df_v2[TARGET])

importance_df_v2 = pd.DataFrame({
    "feature": FEATURES_XGB,
    "importance": model_xgb_v2_final.feature_importances_
}).sort_values("importance", ascending=False)

print(importance_df_v2)

# =========================================================
#  9. Detailed Results Table
# =========================================================

xgb_v2_final_results["error"] = xgb_v2_final_results["prediction"] - xgb_v2_final_results["actual"]
xgb_v2_final_results["abs_error"] = np.abs(xgb_v2_final_results["error"])
xgb_v2_final_results["error_pct"] = (
    xgb_v2_final_results["abs_error"] / xgb_v2_final_results["actual"]
) * 100

print(xgb_v2_final_results)

# =========================================================
#  10. Visualize Results
# =========================================================
# What this step does:
# Plots actual vs predicted values over time.

import matplotlib.pyplot as plt

plt.figure(figsize=(12, 6))
plt.plot(xgb_v2_final_results["date"], xgb_v2_final_results["actual"], label="Actual")
plt.plot(xgb_v2_final_results["date"], xgb_v2_final_results["prediction"], label="Prediction")
plt.legend()
plt.title("XGBoost - Rolling Forecast")
plt.show()

## Final Model Preparation for Forecasting

This section recreates the selected Ridge regression model using a clean and streamlined implementation for use in the forecasting workflow.

The objective is to:

- remove experimental and diagnostic code from earlier development steps
- ensure consistent feature preprocessing
- prepare the model for reproducible forecasting

This version serves as the primary model used in the forecasting notebook.

### Final Model — Ridge Regression (Version 1)


In [ ]:
# =========================================================
#  0. Imports
# =========================================================

import numpy as np
import pandas as pd
import joblib

from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import Ridge
from sklearn.metrics import mean_absolute_error, mean_squared_error

# =========================================================
#  1. Feature groups
# =========================================================

macro_features_2 = [
    "gdp_growth_lag_1",
    "interest_rate_lag_0",
    "cpi_lag_0",
    "real_final_consumption_lag_7",
    "petrol_euro95_lag_4",
    "acea_total_registrations_lag_12"
]

time_features_2 = [
]

season_features = [
    "is_peak_month",
    "is_dip_month"
]

FEATURES_V4 = time_features_2 + macro_features_2 + season_features
TARGET = "acea_total_registrations"

ALPHAS = [0.01, 0.1, 1, 10, 50, 100, 200]
START_DATE = "2024-01-01"
FINAL_TRAIN_END_DATE = "2025-12-01"

PEAK_MONTHS = [3, 6]
DIP_MONTHS = [1, 8]

# =========================================================
#  2. Prepare training dataframe
# =========================================================

TRAIN_DF = full_model_df.copy()
TRAIN_DF["date"] = pd.to_datetime(TRAIN_DF["date"])
TRAIN_DF = TRAIN_DF.sort_values("date").reset_index(drop=True)

# Add missing season flags from date
TRAIN_DF["month"] = TRAIN_DF["date"].dt.month
TRAIN_DF["is_peak_month"] = TRAIN_DF["month"].isin(PEAK_MONTHS).astype(int)
TRAIN_DF["is_dip_month"] = TRAIN_DF["month"].isin(DIP_MONTHS).astype(int)

required_cols = ["date"] + FEATURES_V4 + [TARGET]
missing_cols = [c for c in required_cols if c not in TRAIN_DF.columns]
if missing_cols:
    raise ValueError(f"TRAIN_DF is missing required columns: {missing_cols}")

TRAIN_DF = TRAIN_DF.dropna(subset=required_cols).copy()

print("Training dataframe shape after cleanup:", TRAIN_DF.shape)
display(TRAIN_DF[["date"] + FEATURES_V4 + [TARGET]].tail())

# =========================================================
#  3. Rolling backtest with pipeline
# =========================================================

def rolling_backtest_pipeline(df, start_date, features, target, alpha):
    df = df.copy()
    df["date"] = pd.to_datetime(df["date"])
    df = df.sort_values("date").reset_index(drop=True)

    start_date = pd.to_datetime(start_date)

    predictions = []
    actuals = []
    dates = []

    for i in range(len(df)):
        current_date = df.loc[i, "date"]

        if current_date < start_date:
            continue

        train = df[df["date"] < current_date].copy()
        test = df[df["date"] == current_date].copy()

        if len(train) == 0 or len(test) == 0:
            continue

        X_train = train[features]
        y_train = train[target]

        X_test = test[features]
        y_test = test[target]

        pipeline = Pipeline([
            ("scaler", StandardScaler()),
            ("model", Ridge(alpha=alpha))
        ])

        pipeline.fit(X_train, y_train)
        y_pred = pipeline.predict(X_test)

        predictions.append(float(y_pred[0]))
        actuals.append(float(y_test.values[0]))
        dates.append(current_date)

    results = pd.DataFrame({
        "date": dates,
        "actual": actuals,
        "prediction": predictions
    })

    return results

# =========================================================
#  4. Tune alpha
# =========================================================

ridge_results_summary = []

for alpha in ALPHAS:
    res = rolling_backtest_pipeline(
        df=TRAIN_DF,
        start_date=START_DATE,
        features=FEATURES_V4,
        target=TARGET,
        alpha=alpha
    )

    mape = np.mean(
        np.abs((res["actual"] - res["prediction"]) / res["actual"])
    ) * 100

    ridge_results_summary.append({
        "alpha": alpha,
        "MAPE": round(mape, 3)
    })

ridge_summary_df = pd.DataFrame(ridge_results_summary).sort_values("MAPE").reset_index(drop=True)

print("=== Ridge alpha tuning summary ===")
display(ridge_summary_df)

# =========================================================
#  5. Best alpha + final backtest results
# =========================================================

best_alpha = float(ridge_summary_df.iloc[0]["alpha"])
print(f"Best alpha: {best_alpha}")

ridge_v1_results = rolling_backtest_pipeline(
    df=TRAIN_DF,
    start_date=START_DATE,
    features=FEATURES_V4,
    target=TARGET,
    alpha=best_alpha
)

ridge_v1_mae = mean_absolute_error(
    ridge_v1_results["actual"],
    ridge_v1_results["prediction"]
)

ridge_v1_rmse = np.sqrt(mean_squared_error(
    ridge_v1_results["actual"],
    ridge_v1_results["prediction"]
))

ridge_v1_mape = np.mean(
    np.abs(
        (ridge_v1_results["actual"] - ridge_v1_results["prediction"])
        / ridge_v1_results["actual"]
    )
) * 100

print(f"Ridge v1 MAE:  {ridge_v1_mae:,.2f}")
print(f"Ridge v1 RMSE: {ridge_v1_rmse:,.2f}")
print(f"Ridge v1 MAPE: {ridge_v1_mape:.2f}%")
print("=========================================================")

display(ridge_v1_results.head())
display(ridge_v1_results.tail())

# =========================================================
#  6. Fit final pipeline on all available training data
# =========================================================

final_train_df = TRAIN_DF.copy()
final_train_df = final_train_df[final_train_df["date"] <= FINAL_TRAIN_END_DATE].copy()

X_final = final_train_df[FEATURES_V4]
y_final = final_train_df[TARGET]

ridge_pipeline_v1 = Pipeline([
    ("scaler", StandardScaler()),
    ("model", Ridge(alpha=best_alpha))
])

ridge_pipeline_v1.fit(X_final, y_final)

print("Final pipeline fitted.")
print("Training rows used for final fit:", len(final_train_df))



In [ ]:
# =========================================================
#  7. Save pipeline
# =========================================================

joblib.dump(ridge_pipeline_v1, "ridge_pipeline_v1.pkl")
print("Saved: ridge_pipeline_v1.pkl")

### Final Model — Ridge Regression (Version 2)

In [ ]:
import numpy as np
import pandas as pd
import joblib

from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import Ridge

# =========================================================
# 1. SETTINGS
# =========================================================

TARGET = "acea_total_registrations"

FEATURES_RIDGE_V2 = [
    "gdp_growth_lag_1",
    "interest_rate_lag_0",
    "cpi_lag_0",
    "real_final_consumption_lag_7",
    "petrol_euro95_lag_4",
    "vehicle_production_lag_1",
    "is_peak_month",
    "is_dip_month"
]

alphas = [0.01, 0.1, 1, 10, 50, 100]

PEAK_MONTHS = [3, 6]
DIP_MONTHS = [1, 8]

# =========================================================
# 2. PREP DATA
# =========================================================

ridge_df = macro_df_ind.copy()
ridge_df["date"] = pd.to_datetime(ridge_df["date"])
ridge_df = ridge_df.sort_values("date").reset_index(drop=True)

# create season flags if missing
ridge_df["month"] = ridge_df["date"].dt.month

if "is_peak_month" not in ridge_df.columns:
    ridge_df["is_peak_month"] = ridge_df["month"].isin(PEAK_MONTHS).astype(int)

if "is_dip_month" not in ridge_df.columns:
    ridge_df["is_dip_month"] = ridge_df["month"].isin(DIP_MONTHS).astype(int)

# create vehicle_production_lag_1 if missing
if "vehicle_production_lag_1" not in ridge_df.columns:
    if "vehicle_production" not in ridge_df.columns:
        raise ValueError("Neither 'vehicle_production_lag_1' nor raw 'vehicle_production' exists in full_model_df.")
    ridge_df["vehicle_production_lag_1"] = ridge_df["vehicle_production"].shift(1)

# check required columns
required_cols = ["date"] + FEATURES_RIDGE_V2 + [TARGET]
missing_cols = [c for c in required_cols if c not in ridge_df.columns]
if missing_cols:
    raise ValueError(f"Missing required columns: {missing_cols}")

# drop incomplete rows
ridge_df = ridge_df.dropna(subset=required_cols).copy()

print("Prepared ridge_df shape:", ridge_df.shape)
display(ridge_df[["date"] + FEATURES_RIDGE_V2 + [TARGET]].tail())

# =========================================================
# 3. TUNE ALPHA WITH ROLLING BACKTEST
# =========================================================

ridge_results_summary = []

for alpha in alphas:
    model = Pipeline([
        ("scaler", StandardScaler()),
        ("model", Ridge(alpha=alpha))
    ])

    res = rolling_backtest(
        df=ridge_df,
        start_date="2024-01-01",
        features=FEATURES_RIDGE_V2,
        target=TARGET,
        model=model
    )

    mape = np.mean(
        np.abs((res["actual"] - res["prediction"]) / res["actual"])
    ) * 100

    ridge_results_summary.append({
        "alpha": alpha,
        "MAPE": round(mape, 3)
    })

ridge_summary_df = pd.DataFrame(ridge_results_summary).sort_values("MAPE").reset_index(drop=True)

print("=== Ridge V2 Alpha Tuning ===")
display(ridge_summary_df)

# =========================================================
# 4. FIT FINAL PIPELINE
# =========================================================

best_alpha = float(ridge_summary_df.iloc[0]["alpha"])
print("Best alpha:", best_alpha)

final_train_df = ridge_df.copy()
final_train_df = final_train_df[final_train_df["date"] <= "2025-12-01"].copy()

X_final = final_train_df[FEATURES_RIDGE_V2]
y_final = final_train_df[TARGET]

ridge_v2_pipeline = Pipeline([
    ("scaler", StandardScaler()),
    ("model", Ridge(alpha=best_alpha))
])

ridge_v2_pipeline.fit(X_final, y_final)

print("Final Ridge V2 pipeline fitted.")
print("Training rows used:", len(final_train_df))

# =========================================================
# 5. SAVE PIPELINE
# =========================================================

joblib.dump(ridge_v2_pipeline, "ridge_v2_pipeline.pkl")
print("Saved: ridge_v2_pipeline.pkl")

### Final Model — XGBoost

A clean version of the XGBoost model is prepared to enable comparison with the Ridge-based approach during forecasting.

In [ ]:
# =========================================================
#  0. Imports
# =========================================================

import numpy as np
import pandas as pd
import joblib

from xgboost import XGBRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error

# =========================================================
#  1. Prepare Dataset
# =========================================================

# Use the same dataset you used before for XGBoost
xgb_df = macro_df_ind.copy()

TARGET = "acea_total_registrations"

FEATURES_XGB = [
    "vehicle_production_lag_1",
    "gdp_growth_lag_1",
    "interest_rate_lag_0",
    "cpi_lag_0",
    "real_final_consumption_lag_7",
    "petrol_euro95_lag_4",
    "is_peak_month",
    "is_dip_month",
]

xgb_df["date"] = pd.to_datetime(xgb_df["date"])
xgb_df = xgb_df.sort_values("date").reset_index(drop=True)

required_cols = ["date", TARGET] + FEATURES_XGB
missing_cols = [col for col in required_cols if col not in xgb_df.columns]
if missing_cols:
    raise ValueError(f"xgb_df is missing required columns: {missing_cols}")

xgb_df = xgb_df[required_cols].dropna().reset_index(drop=True)

print("Features used:", FEATURES_XGB)
print("Training rows:", len(xgb_df))
print("=========================================================")
display(xgb_df.tail())

# =========================================================
#  2. Rolling Backtest Function
# =========================================================

def rolling_backtest(
    df,
    start_date,
    features,
    target,
    model
):
    df = df.copy()
    df["date"] = pd.to_datetime(df["date"])
    df = df.sort_values("date").reset_index(drop=True)

    start_date = pd.to_datetime(start_date)

    predictions = []
    actuals = []
    dates = []

    for i in range(len(df)):
        current_date = df.loc[i, "date"]

        if current_date < start_date:
            continue

        train = df[df["date"] < current_date].copy()
        test = df[df["date"] == current_date].copy()

        if len(train) == 0 or len(test) == 0:
            continue

        X_train = train[features]
        y_train = train[target]

        X_test = test[features]
        y_test = test[target]

        model.fit(X_train, y_train)
        y_pred = model.predict(X_test)

        predictions.append(float(y_pred[0]))
        actuals.append(float(y_test.values[0]))
        dates.append(current_date)

    results = pd.DataFrame({
        "date": dates,
        "actual": actuals,
        "prediction": predictions
    })

    return results

# =========================================================
#  3. Define Parameter Grid
# =========================================================

xgb_param_grid = [
    {"n_estimators": 100, "max_depth": 2, "learning_rate": 0.05},
    {"n_estimators": 200, "max_depth": 2, "learning_rate": 0.05},
    {"n_estimators": 100, "max_depth": 3, "learning_rate": 0.05},
    {"n_estimators": 200, "max_depth": 3, "learning_rate": 0.05},
    {"n_estimators": 100, "max_depth": 3, "learning_rate": 0.1},
    {"n_estimators": 200, "max_depth": 3, "learning_rate": 0.1},
    {"n_estimators": 100, "max_depth": 4, "learning_rate": 0.05},
    {"n_estimators": 200, "max_depth": 4, "learning_rate": 0.05},
]

# =========================================================
#  4. Tune XGBoost using Rolling Backtest
# =========================================================

xgb_results_summary = []

for params in xgb_param_grid:

    model_xgb = XGBRegressor(
        n_estimators=params["n_estimators"],
        max_depth=params["max_depth"],
        learning_rate=params["learning_rate"],
        random_state=42,
        objective="reg:squarederror"
    )

    res = rolling_backtest(
        df=xgb_df,
        start_date="2024-01-01",
        features=FEATURES_XGB,
        target=TARGET,
        model=model_xgb
    )

    mape = np.mean(
        np.abs((res["actual"] - res["prediction"]) / res["actual"])
    ) * 100

    xgb_results_summary.append({
        "n_estimators": params["n_estimators"],
        "max_depth": params["max_depth"],
        "learning_rate": params["learning_rate"],
        "MAPE": round(mape, 3)
    })

xgb_summary_df = pd.DataFrame(xgb_results_summary).sort_values("MAPE").reset_index(drop=True)

print("=== XGBoost tuning summary ===")
display(xgb_summary_df)

# =========================================================
#  5. Select Best Parameters
# =========================================================

best_params = xgb_summary_df.iloc[0]

best_n = int(best_params["n_estimators"])
best_depth = int(best_params["max_depth"])
best_lr = float(best_params["learning_rate"])

print("Best parameters:")
print(best_params)
print("=========================================================")

# =========================================================
#  6. Backtest Final XGBoost Model
# =========================================================

model_xgb_v1 = XGBRegressor(
    n_estimators=best_n,
    max_depth=best_depth,
    learning_rate=best_lr,
    random_state=42,
    objective="reg:squarederror"
)

xgb_v1_results = rolling_backtest(
    df=xgb_df,
    start_date="2024-01-01",
    features=FEATURES_XGB,
    target=TARGET,
    model=model_xgb_v1
)

# =========================================================
#  7. Evaluate Performance
# =========================================================

xgb_mae = mean_absolute_error(
    xgb_v1_results["actual"],
    xgb_v1_results["prediction"]
)

xgb_rmse = np.sqrt(mean_squared_error(
    xgb_v1_results["actual"],
    xgb_v1_results["prediction"]
))

xgb_mape = np.mean(
    np.abs(
        (xgb_v1_results["actual"] - xgb_v1_results["prediction"])
        / xgb_v1_results["actual"]
    )
) * 100

print(f"XGBoost MAE: {xgb_mae:,.2f}")
print(f"XGBoost RMSE: {xgb_rmse:,.2f}")
print(f"XGBoost MAPE: {xgb_mape:.2f}%")
print("=========================================================")

display(xgb_v1_results.head())
display(xgb_v1_results.tail())

# =========================================================
#  8. Fit Final Saved Model on Full Historical Data
# =========================================================
# This is the model you will load later in the forecasting notebook

final_train_df = xgb_df.copy()
final_train_df = final_train_df[final_train_df["date"] <= "2025-12-01"].copy()

X_final = final_train_df[FEATURES_XGB]
y_final = final_train_df[TARGET]

model_xgb_v1_final = XGBRegressor(
    n_estimators=best_n,
    max_depth=best_depth,
    learning_rate=best_lr,
    random_state=42,
    objective="reg:squarederror"
)

model_xgb_v1_final.fit(X_final, y_final)

print("Final XGBoost model fitted.")
print("Training rows used for final fit:", len(final_train_df))

# =========================================================
#  9. Save Final Model
# =========================================================

joblib.dump(model_xgb_v1_final, "xgb_model_v1.pkl")
print("Saved: xgb_model_v1.pkl")

# =========================================================
# 10. Feature Importance from Final Saved Model
# =========================================================

importance_df = pd.DataFrame({
    "feature": FEATURES_XGB,
    "importance": model_xgb_v1_final.feature_importances_
}).sort_values("importance", ascending=False)

print("=== XGBoost Feature Importance ===")
display(importance_df)

# =========================================================
# 11. Optional Model Comparison
# =========================================================

print("Models Comparison:")
print(f"Baseline MAPE: {baseline_mape:.2f}%")
print(f"Macro v3 MAPE: {macro_v3_mape:.2f}%")
print(f"Macro v4 MAPE: {macro_v4_mape:.2f}%")
print(f"ridge_v1_mape: {ridge_v1_mape:.2f}%")
print(f"ridge_v5_mape: {ridge_v5_mape:.2f}%")
print(f"Random Forest MAPE: {rf_final_mape:.2f}%")
print(f"XGBoost MAPE: {xgb_mape:.2f}%")
print("=========================================================")